<a href="https://colab.research.google.com/github/TralAlex/ITEH-Domaci1/blob/main/Projekat1MLBigData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pyspark kagglehub

In [ ]:
import os
import re
import glob
import kagglehub

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
# Download latest version
path = kagglehub.dataset_download("bromotdi/movielens-100k-dataset")
print("Path to dataset files:", path)
print("Files:", os.listdir(path))

100%|██████████| 2.53M/2.53M [00:00<00:00, 127MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1
Files: ['ml-100k']


In [ ]:
spark = (
    SparkSession.builder
    .appName("MovieLens100K-Recommender")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
import os
import glob

def find_file_safe(base_path, filename):
    matches = glob.glob(os.path.join(base_path, "**", filename), recursive=True)
    return matches[0] if matches else None

u_data_path = find_file_safe(path, "u.data")
u_item_path = find_file_safe(path, "u.item")
u_user_path = find_file_safe(path, "u.user")
u1_base_path = find_file_safe(path, "u1.base")
u1_test_path = find_file_safe(path, "u1.test")
u2_base_path = find_file_safe(path, "u2.base")

print("u.data  ->", u_data_path)
print("u.item  ->", u_item_path)
print("u.user  ->", u_user_path)
print("u1.base ->", u1_base_path)
print("u1.test ->", u1_test_path)
print("u2.base ->", u2_base_path)

# fallback ako ne postoji
if u2_base_path is None:
    print("⚠️ u2.base ne postoji → koristićemo u.data kao zamenu")
    u2_base_path = u_data_path

u.data  -> /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1/ml-100k/u.data
u.item  -> /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1/ml-100k/u.item
u.user  -> /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1/ml-100k/u.user
u1.base -> /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1/ml-100k/u1.base
u1.test -> /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1/ml-100k/u1.test
u2.base -> None
⚠️ u2.base ne postoji → koristićemo u.data kao zamenu


In [ ]:
ratings_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("movie_id", IntegerType(), True),
    StructField("rating", IntegerType(), True),
    StructField("timestamp", IntegerType(), True),
])

users_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("occupation", StringType(), True),
    StructField("zip_code", StringType(), True),
])

# ratings
ratings = spark.read.csv(
    u_data_path,
    sep="\t",
    schema=ratings_schema,
    header=False
)

u1_base = spark.read.csv(
    u1_base_path,
    sep="\t",
    schema=ratings_schema,
    header=False
)

u1_test = spark.read.csv(
    u1_test_path,
    sep="\t",
    schema=ratings_schema,
    header=False
)

u2_base = spark.read.csv(
    u2_base_path,
    sep="\t",
    schema=ratings_schema,
    header=False
)

# users
users = spark.read.csv(
    u_user_path,
    sep="|",
    schema=users_schema,
    header=False
)

# movies (u.item)
movies_raw = spark.read.text(u_item_path)
movies_raw = movies_raw.select(F.col("value"))
movies_raw.show(3, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                        |
+-----------------------------------------------------------------------------------------------------------------------------+
|1|Toy Story (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Toy%20Story%20(1995)|0|0|0|1|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0  |
|2|GoldenEye (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?GoldenEye%20(1995)|0|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0    |
|3|Four Rooms (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Four%20Rooms%20(1995)|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0|
+-----------------------------------------------------------------------------------------------------------------------------+
only showing top 3 rows


In [ ]:
genre_names = [
    "unknown", "Action", "Adventure", "Animation", "Childrens", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "FilmNoir", "Horror",
    "Musical", "Mystery", "Romance", "SciFi", "Thriller", "War", "Western"
]

def parse_u_item_line(line):
    parts = line.split("|")
    if len(parts) < 24:
        parts = parts + ["0"] * (24 - len(parts))

    movie_id = int(parts[0])
    title = parts[1]
    release_date = parts[2]
    imdb_url = parts[4]

    genre_flags = []
    for x in parts[5:24]:
        try:
            genre_flags.append(int(x))
        except:
            genre_flags.append(0)

    return tuple([movie_id, title, release_date, imdb_url] + genre_flags)

movie_schema = StructType(
    [
        StructField("movie_id", IntegerType(), True),
        StructField("title", StringType(), True),
        StructField("release_date", StringType(), True),
        StructField("imdb_url", StringType(), True),
    ] +
    [StructField(g, IntegerType(), True) for g in genre_names]
)

movies_rdd = movies_raw.rdd.map(lambda r: parse_u_item_line(r["value"]))
movies = spark.createDataFrame(movies_rdd, schema=movie_schema)

movies = movies.withColumn(
    "movie_year_str",
    F.regexp_extract(F.col("title"), r"\((\d{4})\)", 1)
)

movies = movies.withColumn(
    "movie_year",
    F.when(F.col("movie_year_str") != "", F.col("movie_year_str").cast("int"))
     .otherwise(None)
).drop("movie_year_str")

# glavni žanr = prvi genre flag koji je 1
genre_expr = (
    F.when(F.col("unknown") == 1, "unknown")
     .when(F.col("Action") == 1, "Action")
     .when(F.col("Adventure") == 1, "Adventure")
     .when(F.col("Animation") == 1, "Animation")
     .when(F.col("Childrens") == 1, "Childrens")
     .when(F.col("Comedy") == 1, "Comedy")
     .when(F.col("Crime") == 1, "Crime")
     .when(F.col("Documentary") == 1, "Documentary")
     .when(F.col("Drama") == 1, "Drama")
     .when(F.col("Fantasy") == 1, "Fantasy")
     .when(F.col("FilmNoir") == 1, "FilmNoir")
     .when(F.col("Horror") == 1, "Horror")
     .when(F.col("Musical") == 1, "Musical")
     .when(F.col("Mystery") == 1, "Mystery")
     .when(F.col("Romance") == 1, "Romance")
     .when(F.col("SciFi") == 1, "SciFi")
     .when(F.col("Thriller") == 1, "Thriller")
     .when(F.col("War") == 1, "War")
     .when(F.col("Western") == 1, "Western")
     .otherwise("unknown")
)

movies = movies.withColumn("genre", genre_expr)

movies.select("movie_id", "title", "movie_year", "genre").show(10, truncate=False)

+--------+----------------------------------------------------+----------+---------+
|movie_id|title                                               |movie_year|genre    |
+--------+----------------------------------------------------+----------+---------+
|1       |Toy Story (1995)                                    |1995      |Animation|
|2       |GoldenEye (1995)                                    |1995      |Action   |
|3       |Four Rooms (1995)                                   |1995      |Thriller |
|4       |Get Shorty (1995)                                   |1995      |Action   |
|5       |Copycat (1995)                                      |1995      |Crime    |
|6       |Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)|1995      |Drama    |
|7       |Twelve Monkeys (1995)                               |1995      |Drama    |
|8       |Babe (1995)                                         |1995      |Childrens|
|9       |Dead Man Walking (1995)                             |19

In [ ]:
movie_stats = (
    ratings.groupBy("movie_id")
    .agg(
        F.count("*").alias("view_count"),
        F.avg("rating").alias("avg_rating")
    )
    .join(movies.select("movie_id", "title", "genre", "movie_year"), on="movie_id", how="left")
)

top_50_popular = (
    movie_stats.orderBy(F.desc("view_count"), F.desc("avg_rating"))
    .limit(50)
)

print("Top 50 najpopularnijih filmova:")
top_50_popular.select("movie_id", "title", "view_count", F.round("avg_rating", 3).alias("avg_rating")).show(50, truncate=False)

print("10 filmova sa NAJVISE pregleda:")
movie_stats.orderBy(F.desc("view_count"), F.desc("avg_rating")).select("title", "view_count", F.round("avg_rating", 3).alias("avg_rating")).show(10, truncate=False)

print("10 filmova sa NAJMANJE pregleda:")
movie_stats.orderBy(F.asc("view_count"), F.asc("avg_rating")).select("title", "view_count", F.round("avg_rating", 3).alias("avg_rating")).show(10, truncate=False)

Top 50 najpopularnijih filmova:
+--------+--------------------------------------------+----------+----------+
|movie_id|title                                       |view_count|avg_rating|
+--------+--------------------------------------------+----------+----------+
|50      |Star Wars (1977)                            |583       |4.358     |
|258     |Contact (1997)                              |509       |3.804     |
|100     |Fargo (1996)                                |508       |4.156     |
|181     |Return of the Jedi (1983)                   |507       |4.008     |
|294     |Liar Liar (1997)                            |485       |3.157     |
|286     |English Patient, The (1996)                 |481       |3.657     |
|288     |Scream (1996)                               |478       |3.441     |
|1       |Toy Story (1995)                            |452       |3.878     |
|300     |Air Force One (1997)                        |431       |3.631     |
|121     |Independence Day (ID4)

In [ ]:
max_vals = movie_stats.agg(
    F.max("view_count").alias("max_view_count"),
    F.max("avg_rating").alias("max_avg_rating")
).collect()[0]

max_view_count = max_vals["max_view_count"]
max_avg_rating = max_vals["max_avg_rating"]

movie_scores = movie_stats.withColumn(
    "score",
    (F.col("view_count") * F.col("avg_rating")) / F.lit(max_view_count * max_avg_rating)
)

print("Top 5 filmova po score-u:")
movie_scores.orderBy(F.desc("score")).select(
    "title", "view_count", F.round("avg_rating", 3).alias("avg_rating"), F.round("score", 5).alias("score")
).show(5, truncate=False)

Top 5 filmova po score-u:
+------------------------------+----------+----------+-------+
|title                         |view_count|avg_rating|score  |
+------------------------------+----------+----------+-------+
|Star Wars (1977)              |583       |4.358     |0.8717 |
|Fargo (1996)                  |508       |4.156     |0.72419|
|Return of the Jedi (1983)     |507       |4.008     |0.69708|
|Contact (1997)                |509       |3.804     |0.66415|
|Raiders of the Lost Ark (1981)|420       |4.252     |0.61269|
+------------------------------+----------+----------+-------+
only showing top 5 rows


In [ ]:
ratings_users_movies = (
    ratings.join(users, on="user_id", how="left")
           .join(movies.select("movie_id", "title", "genre", "movie_year"), on="movie_id", how="left")
)

gender_movie_stats = (
    ratings_users_movies.groupBy("gender", "movie_id", "title")
    .agg(
        F.count("*").alias("view_count"),
        F.avg("rating").alias("avg_rating")
    )
)

w_gender = Window.partitionBy("gender").orderBy(F.desc("view_count"), F.desc("avg_rating"))

top5_by_gender = (
    gender_movie_stats
    .withColumn("rank", F.row_number().over(w_gender))
    .filter(F.col("rank") <= 5)
    .orderBy("gender", "rank")
)

print("5 najpopularnijih filmova po polu:")
top5_by_gender.show(20, truncate=False)

5 najpopularnijih filmova po polu:
+------+--------+---------------------------+----------+------------------+----+
|gender|movie_id|title                      |view_count|avg_rating        |rank|
+------+--------+---------------------------+----------+------------------+----+
|F     |286     |English Patient, The (1996)|152       |3.8092105263157894|1   |
|F     |50      |Star Wars (1977)           |151       |4.245033112582782 |2   |
|F     |288     |Scream (1996)              |143       |3.3426573426573425|3   |
|F     |294     |Liar Liar (1997)           |141       |3.099290780141844 |4   |
|F     |258     |Contact (1997)             |137       |3.6861313868613137|5   |
|M     |50      |Star Wars (1977)           |432       |4.398148148148148 |1   |
|M     |100     |Fargo (1996)               |383       |4.201044386422977 |2   |
|M     |181     |Return of the Jedi (1983)  |383       |4.007832898172324 |3   |
|M     |258     |Contact (1997)             |372       |3.846774193548387 

In [ ]:
gender_occ_movie_stats = (
    ratings_users_movies.groupBy("gender", "occupation", "movie_id", "title")
    .agg(
        F.count("*").alias("view_count"),
        F.avg("rating").alias("avg_rating")
    )
)

w_gender_occ = Window.partitionBy("gender", "occupation").orderBy(F.desc("view_count"), F.desc("avg_rating"))

top3_by_gender_occ = (
    gender_occ_movie_stats
    .withColumn("rank", F.row_number().over(w_gender_occ))
    .filter(F.col("rank") <= 3)
    .orderBy("gender", "occupation", "rank")
)

print("3 najpopularnija filma po polu i zanimanju:")
top3_by_gender_occ.show(100, truncate=False)

3 najpopularnija filma po polu i zanimanju:
+------+-------------+--------+--------------------------------------------+----------+------------------+----+
|gender|occupation   |movie_id|title                                       |view_count|avg_rating        |rank|
+------+-------------+--------+--------------------------------------------+----------+------------------+----+
|F     |administrator|286     |English Patient, The (1996)                 |21        |4.285714285714286 |1   |
|F     |administrator|50      |Star Wars (1977)                            |21        |4.190476190476191 |2   |
|F     |administrator|237     |Jerry Maguire (1996)                        |19        |4.0               |3   |
|F     |artist       |258     |Contact (1997)                              |10        |3.6               |1   |
|F     |artist       |127     |Godfather, The (1972)                       |8         |4.5               |2   |
|F     |artist       |50      |Star Wars (1977)             

In [ ]:
genre_movie_stats = (
    ratings.join(movies.select("movie_id", "title", "genre"), on="movie_id", how="left")
    .groupBy("genre", "movie_id", "title")
    .agg(
        F.count("*").alias("view_count"),
        F.avg("rating").alias("avg_rating")
    )
)

w_genre = Window.partitionBy("genre").orderBy(F.desc("view_count"), F.desc("avg_rating"))

top3_by_genre = (
    genre_movie_stats
    .withColumn("rank", F.row_number().over(w_genre))
    .filter(F.col("rank") <= 3)
    .orderBy("genre", "rank")
)

print("3 najpopularnija filma u svakom žanru:")
top3_by_genre.show(100, truncate=False)

3 najpopularnija filma u svakom žanru:
+-----------+--------+---------------------------------------------------------------------------+----------+------------------+----+
|genre      |movie_id|title                                                                      |view_count|avg_rating        |rank|
+-----------+--------+---------------------------------------------------------------------------+----------+------------------+----+
|Action     |50      |Star Wars (1977)                                                           |583       |4.3584905660377355|1   |
|Action     |181     |Return of the Jedi (1983)                                                  |507       |4.007889546351085 |2   |
|Action     |300     |Air Force One (1997)                                                       |431       |3.6310904872389793|3   |
|Adventure  |151     |Willy Wonka and the Chocolate Factory (1971)                               |326       |3.6319018404907975|1   |
|Adventure  |97      |D

In [ ]:
users_with_age_group = users.withColumn(
    "age_group",
    F.when(F.col("age") < 18, "under_18")
     .when((F.col("age") >= 18) & (F.col("age") <= 24), "18_24")
     .when((F.col("age") >= 25) & (F.col("age") <= 34), "25_34")
     .when((F.col("age") >= 35) & (F.col("age") <= 44), "35_44")
     .when((F.col("age") >= 45) & (F.col("age") <= 49), "45_49")
     .when((F.col("age") >= 50) & (F.col("age") <= 55), "50_55")
     .otherwise("56_plus")
)

agegroup_movie_stats = (
    ratings.join(users_with_age_group, on="user_id", how="left")
    .join(movies.select("movie_id", "title"), on="movie_id", how="left")
    .groupBy("age_group", "movie_id", "title")
    .agg(
        F.count("*").alias("view_count"),
        F.avg("rating").alias("avg_rating")
    )
)

w_age = Window.partitionBy("age_group").orderBy(F.desc("view_count"), F.desc("avg_rating"))

top5_by_age_group = (
    agegroup_movie_stats
    .withColumn("rank", F.row_number().over(w_age))
    .filter(F.col("rank") <= 5)
    .orderBy("age_group", "rank")
)

print("5 najpopularnijih filmova za svaku starosnu grupu:")
top5_by_age_group.show(100, truncate=False)

5 najpopularnijih filmova za svaku starosnu grupu:
+---------+--------+---------------------------+----------+------------------+----+
|age_group|movie_id|title                      |view_count|avg_rating        |rank|
+---------+--------+---------------------------+----------+------------------+----+
|18_24    |288     |Scream (1996)              |148       |3.695945945945946 |1   |
|18_24    |50      |Star Wars (1977)           |126       |4.4523809523809526|2   |
|18_24    |181     |Return of the Jedi (1983)  |120       |4.308333333333334 |3   |
|18_24    |258     |Contact (1997)             |119       |3.689075630252101 |4   |
|18_24    |294     |Liar Liar (1997)           |119       |3.46218487394958  |5   |
|25_34    |50      |Star Wars (1977)           |222       |4.400900900900901 |1   |
|25_34    |181     |Return of the Jedi (1983)  |188       |3.877659574468085 |2   |
|25_34    |1       |Toy Story (1995)           |181       |3.9502762430939224|3   |
|25_34    |100     |Fargo

In [ ]:
overall_avg_rating = ratings.agg(F.avg("rating").alias("overall_avg_rating"))
print("Ukupna prosečna ocena svih filmova:")
overall_avg_rating.show()

Ukupna prosečna ocena svih filmova:
+------------------+
|overall_avg_rating|
+------------------+
|           3.52986|
+------------------+



In [ ]:
u1_train_cb = (
    u1_base.join(users, on="user_id", how="left")
           .join(movies.select("movie_id", "title", "movie_year", "genre"), on="movie_id", how="left")
           .select("user_id", "movie_id", "rating", "age", "gender", "occupation", "genre", "movie_year")
           .dropna()
)

u1_test_cb = (
    u1_test.join(users, on="user_id", how="left")
           .join(movies.select("movie_id", "title", "movie_year", "genre"), on="movie_id", how="left")
           .select("user_id", "movie_id", "rating", "age", "gender", "occupation", "genre", "movie_year")
           .dropna()
)

u1_train_cb.show(5, truncate=False)

+-------+--------+------+---+------+-------------+---------+----------+
|user_id|movie_id|rating|age|gender|occupation   |genre    |movie_year|
+-------+--------+------+---+------+-------------+---------+----------+
|6      |496     |4     |42 |M     |executive    |Drama    |1946      |
|7      |463     |4     |57 |M     |administrator|Adventure|1994      |
|13     |463     |5     |47 |M     |educator     |Adventure|1994      |
|13     |471     |1     |47 |M     |educator     |Drama    |1996      |
|13     |833     |2     |47 |M     |educator     |Action   |1996      |
+-------+--------+------+---+------+-------------+---------+----------+
only showing top 5 rows


In [ ]:
cat_cols = ["gender", "occupation", "genre"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in cat_cols]

assembler = VectorAssembler(
    inputCols=["age", "movie_year"] + [f"{c}_ohe" for c in cat_cols],
    outputCol="features"
)

base_stages = indexers + encoders + [assembler]

In [ ]:
lr = LinearRegression(featuresCol="features", labelCol="rating", predictionCol="prediction")

lr_pipeline = Pipeline(stages=base_stages + [lr])

lr_param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

evaluator_rmse = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="r2")

lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    parallelism=2
)

lr_cv_model = lr_cv.fit(u1_train_cb)
lr_preds = lr_cv_model.transform(u1_test_cb)

print("Linear Regression metrics:")
print("RMSE:", evaluator_rmse.evaluate(lr_preds))
print("MAE :", evaluator_mae.evaluate(lr_preds))
print("R2  :", evaluator_r2.evaluate(lr_preds))

Linear Regression metrics:
RMSE: 1.1094338270658786
MAE : 0.916313826948803
R2  : 0.07534799253776037


In [ ]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="rating",
    predictionCol="prediction",
    seed=42
)

rf_pipeline = Pipeline(stages=base_stages + [rf])

rf_param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    parallelism=2
)

rf_cv_model = rf_cv.fit(u1_train_cb)
rf_preds = rf_cv_model.transform(u1_test_cb)

print("Random Forest metrics:")
print("RMSE:", evaluator_rmse.evaluate(rf_preds))
print("MAE :", evaluator_mae.evaluate(rf_preds))
print("R2  :", evaluator_r2.evaluate(rf_preds))

Random Forest metrics:
RMSE: 1.07881583432682
MAE : 0.8799888909540975
R2  : 0.1256805612896903


In [ ]:
lr_rmse = evaluator_rmse.evaluate(lr_preds)
rf_rmse = evaluator_rmse.evaluate(rf_preds)

if lr_rmse <= rf_rmse:
    best_cb_model = lr_cv_model
    best_cb_name = "Linear Regression"


In [ ]:
movies = movies.drop("movie_year") if "movie_year" in movies.columns else movies

movies = movies.withColumn(
    "movie_year_str",
    F.regexp_extract(F.col("title"), r"\((\d{4})\)", 1)
)

movies = movies.withColumn(
    "movie_year",
    F.when(F.col("movie_year_str") != "", F.col("movie_year_str").cast("int"))
     .otherwise(F.lit(None).cast("int"))
).drop("movie_year_str")

movies.select("movie_id", "title", "movie_year", "genre").show(10, truncate=False)

+--------+----------------------------------------------------+----------+---------+
|movie_id|title                                               |movie_year|genre    |
+--------+----------------------------------------------------+----------+---------+
|1       |Toy Story (1995)                                    |1995      |Animation|
|2       |GoldenEye (1995)                                    |1995      |Action   |
|3       |Four Rooms (1995)                                   |1995      |Thriller |
|4       |Get Shorty (1995)                                   |1995      |Action   |
|5       |Copycat (1995)                                      |1995      |Crime    |
|6       |Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)|1995      |Drama    |
|7       |Twelve Monkeys (1995)                               |1995      |Drama    |
|8       |Babe (1995)                                         |1995      |Childrens|
|9       |Dead Man Walking (1995)                             |19

In [ ]:
movie_stats = (
    ratings.groupBy("movie_id")
    .agg(
        F.count("*").alias("view_count"),
        F.avg("rating").alias("avg_rating")
    )
    .join(movies.select("movie_id", "title", "genre", "movie_year"), on="movie_id", how="left")
)

top_50_popular = (
    movie_stats.orderBy(F.desc("view_count"), F.desc("avg_rating"))
    .limit(50)
)

top_50_popular.show(5, truncate=False)

+--------+----------+------------------+-------------------------+------+----------+
|movie_id|view_count|avg_rating        |title                    |genre |movie_year|
+--------+----------+------------------+-------------------------+------+----------+
|50      |583       |4.3584905660377355|Star Wars (1977)         |Action|1977      |
|258     |509       |3.8035363457760316|Contact (1997)           |Drama |1997      |
|100     |508       |4.155511811023622 |Fargo (1996)             |Crime |1996      |
|181     |507       |4.007889546351085 |Return of the Jedi (1983)|Action|1983      |
|294     |485       |3.156701030927835 |Liar Liar (1997)         |Comedy|1997      |
+--------+----------+------------------+-------------------------+------+----------+
only showing top 5 rows


In [ ]:
try:
    best_cb_model
except NameError:
    best_cb_model = lr_cv_model
    best_cb_name = "Linear Regression"

print(f"Best content-based model: {best_cb_name}")

def recommend_top50_content_based(age, gender, occupation, top50_df, best_model, top_n=10):
    user_rows = (
        top50_df
        .select("movie_id", "title", "genre", "movie_year")
        .withColumn("movie_year", F.coalesce(F.col("movie_year"), F.lit(1995)))
        .withColumn("age", F.lit(age))
        .withColumn("gender", F.lit(gender))
        .withColumn("occupation", F.lit(occupation))
    )

    preds = (
        best_model.transform(user_rows)
        .select(
            "movie_id",
            "title",
            "genre",
            "movie_year",
            F.round("prediction", 3).alias("predicted_rating")
        )
        .orderBy(F.desc("predicted_rating"))
    )

    return preds.limit(top_n)

cb_recs = recommend_top50_content_based(
    age=25,
    gender="M",
    occupation="student",
    top50_df=top_50_popular,
    best_model=best_cb_model,
    top_n=10
)

cb_recs.show(truncate=False)

Best content-based model: Linear Regression
+--------+--------------------------------------------+---------+----------+----------------+
|movie_id|title                                       |genre    |movie_year|predicted_rating|
+--------+--------------------------------------------+---------+----------+----------------+
|191     |Amadeus (1984)                              |Drama    |1984      |3.759           |
|127     |Godfather, The (1972)                       |Action   |1972      |3.756           |
|234     |Jaws (1975)                                 |Action   |1975      |3.713           |
|56      |Pulp Fiction (1994)                         |Crime    |1994      |3.702           |
|50      |Star Wars (1977)                            |Action   |1977      |3.685           |
|100     |Fargo (1996)                                |Crime    |1996      |3.674           |
|98      |Silence of the Lambs, The (1991)            |Drama    |1991      |3.661           |
|302     |L.A. C

In [ ]:
# import ALS modela i evaluacije
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
# 🔹 instalacija (ako treba)
!pip -q install pyspark kagglehub

# 🔹 importi
import kagglehub
import os, glob

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 🔹 spark
spark = SparkSession.builder.appName("ALS").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 🔹 download dataset
path = kagglehub.dataset_download("bromotdi/movielens-100k-dataset")
print("Dataset path:", path)

# 🔹 helper za fajlove
def find_file(base_path, filename):
    matches = glob.glob(os.path.join(base_path, "**", filename), recursive=True)
    return matches[0] if matches else None

# 🔹 putanje
u1_base_path = find_file(path, "u1.base")
u1_test_path = find_file(path, "u1.test")

print("u1.base:", u1_base_path)
print("u1.test:", u1_test_path)

# 🔹 schema
ratings_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("movie_id", IntegerType(), True),
    StructField("rating", IntegerType(), True),
    StructField("timestamp", IntegerType(), True),
])

# 🔹 učitavanje
u1_base = spark.read.csv(u1_base_path, sep="\t", schema=ratings_schema)
u1_test = spark.read.csv(u1_test_path, sep="\t", schema=ratings_schema)

u1_base.show(5)

100%|██████████| 2.53M/2.53M [00:00<00:00, 67.7MB/s]

Extracting files...
Dataset path: /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1
u1.base: /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1/ml-100k/u1.base
u1.test: /root/.cache/kagglehub/datasets/bromotdi/movielens-100k-dataset/versions/1/ml-100k/u1.test


+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|      1|       1|     5|874965758|
|      1|       2|     3|876893171|
|      1|       3|     4|878542960|
|      1|       4|     3|876893119|
|      1|       5|     3|889751712|
+-------+--------+------+---------+
only showing top 5 rows


In [ ]:
# priprema podataka za ALS model
als_train = u1_base.select(
    F.col("user_id").cast("int"),
    F.col("movie_id").cast("int"),
    F.col("rating").cast("float")
)

als_test = u1_test.select(
    F.col("user_id").cast("int"),
    F.col("movie_id").cast("int"),
    F.col("rating").cast("float")
)

als_train.show(5)

+-------+--------+------+
|user_id|movie_id|rating|
+-------+--------+------+
|      1|       1|   5.0|
|      1|       2|   3.0|
|      1|       3|   4.0|
|      1|       4|   3.0|
|      1|       5|   3.0|
+-------+--------+------+
only showing top 5 rows


In [ ]:
# import ALS modela, evaluatora i cross-validacije
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
# metrike za procenu modela
evaluator_rmse = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="r2")

In [ ]:
# definisanje ALS modela (bez predictionCol!)
als = ALS(
    userCol="user_id",
    itemCol="movie_id",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    seed=42
)

# grid hiperparametara
als_param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank, [5, 10])        # manji model
    .addGrid(als.maxIter, [5, 10])     # manje iteracija
    .addGrid(als.regParam, [0.1, 0.2]) # jača regularizacija
    .build()
)

# cross-validation
als_cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=als_param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    parallelism=2
)

als_cv_model = als_cv.fit(als_train)
best_als_model = als_cv_model.bestModel

NameError: name 'ALS' is not defined

In [ ]:
als_test_preds = best_als_model.transform(als_test)
als_test_preds.show(10)

+-------+--------+------+----------+
|user_id|movie_id|rating|prediction|
+-------+--------+------+----------+
|    148|       1|   4.0|   3.99364|
|    148|       7|   5.0| 3.7932115|
|    148|      70|   5.0|  3.000896|
|    148|      71|   5.0|  3.310871|
|    148|      78|   1.0| 1.5148673|
|    148|      98|   3.0| 4.0822706|
|    148|     114|   5.0| 4.4144506|
|    148|     116|   5.0| 2.9058156|
|    148|     140|   1.0| 2.6169121|
|    148|     163|   4.0|  3.565174|
+-------+--------+------+----------+
only showing top 10 rows


In [ ]:
# predikcije nad test skupom u1.test
als_test_preds = best_als_model.transform(als_test)

print("ALS performanse na u1.test:")
print("RMSE:", evaluator_rmse.evaluate(als_test_preds))
print("MAE :", evaluator_mae.evaluate(als_test_preds))
print("R2  :", evaluator_r2.evaluate(als_test_preds))

als_test_preds.show(10, truncate=False)

NameError: name 'best_als_model' is not defined

In [ ]:
# poređenje greške na trening i test skupu
als_train_preds = best_als_model.transform(als_train)

train_rmse = evaluator_rmse.evaluate(als_train_preds)
test_rmse = evaluator_rmse.evaluate(als_test_preds)

print("Train RMSE:", train_rmse)
print("Test RMSE :", test_rmse)

if train_rmse + 0.15 < test_rmse:
    print("Zaključak: moguće je pretreniranje modela.")
elif abs(train_rmse - test_rmse) < 0.1:
    print("Zaključak: model je dobro izbalansiran.")
else:
    print("Zaključak: model pokazuje slabiju generalizaciju ili blago podtreniranje.")

Train RMSE: 0.8156605161914128
Test RMSE : 0.9333963016059912
Zaključak: model pokazuje slabiju generalizaciju ili blago podtreniranje.


In [ ]:
# primer preporuke za korisnika 1
user_id_example = 1

user_df = spark.createDataFrame([(user_id_example,)], ["user_id"])

recommendations = best_als_model.recommendForUserSubset(user_df, 10)

recommendations.show(truncate=False)

+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                            |
+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1      |[{1643, 4.976445}, {1449, 4.9247136}, {1651, 4.7690673}, {1650, 4.7690673}, {1645, 4.7690673}, {1636, 4.7690673}, {913, 4.7690673}, {1467, 4.7673984}, {1368, 4.7607455}, {119, 4.6814995}]|
+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [ ]:
# mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# putanja gde ćeš čuvati fajlove
save_path = "/content/drive/MyDrive/ML_Recommender_Project"

import os
os.makedirs(save_path, exist_ok=True)

print("Fajlovi će biti sačuvani u:", save_path)

Fajlovi će biti sačuvani u: /content/drive/MyDrive/ML_Recommender_Project
